# 05 — Innovation Module Evaluation

Evaluates the impact of the three innovation modules:

1. **Prefix Rules** — Uses common English prefixes (un-, re-, pre-, dis-, ...) to guess POS
2. **Web Token Rules** — Handles URLs, emails, hashtags, @mentions, emoji
3. **Compound Context** — Multi-word expression detection (proper noun bigrams, compound adpositions)

Compares **baseline** (without innovation) vs **enhanced** (with innovation) accuracy.

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.parser import parse_conllu, get_forms_and_tags
from src.lexicon import Lexicon
from src.tagger import RuleBasedTagger
from src.evaluate import Evaluator, UPOS_TAGS

plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style('whitegrid')

In [9]:
# Load data
train_sents = parse_conllu('../data/en_ewt-ud-train.conllu')
dev_sents = parse_conllu('../data/en_ewt-ud-dev.conllu')
test_sents = parse_conllu('../data/en_ewt-ud-test.conllu')

lexicon = Lexicon().build(train_sents)
dev_corpus = get_forms_and_tags(dev_sents)
test_corpus = get_forms_and_tags(test_sents)

## 1. Baseline vs Enhanced Accuracy

In [10]:
# Baseline tagger
tagger_base = RuleBasedTagger(lexicon, use_innovation=False)
dev_tagged_base = tagger_base.tag_corpus(dev_corpus)
test_tagged_base = tagger_base.tag_corpus(test_corpus)

# Enhanced tagger (with innovation)
tagger_innov = RuleBasedTagger(lexicon, use_innovation=True)
dev_tagged_innov = tagger_innov.tag_corpus(dev_corpus)
test_tagged_innov = tagger_innov.tag_corpus(test_corpus)

results = {
    'Dev Baseline': Evaluator.accuracy(dev_tagged_base),
    'Dev Enhanced': Evaluator.accuracy(dev_tagged_innov),
    'Test Baseline': Evaluator.accuracy(test_tagged_base),
    'Test Enhanced': Evaluator.accuracy(test_tagged_innov),
}

for name, acc in results.items():
    print(f'{name:>16}: {acc:.4f} ({acc*100:.2f}%)')

print(f'\nDev  improvement: {(results["Dev Enhanced"] - results["Dev Baseline"])*100:+.2f}%')
print(f'Test improvement: {(results["Test Enhanced"] - results["Test Baseline"])*100:+.2f}%')

    Dev Baseline: 0.8722 (87.22%)
    Dev Enhanced: 0.8934 (89.34%)
   Test Baseline: 0.8763 (87.63%)
   Test Enhanced: 0.8953 (89.53%)

Dev  improvement: +2.12%
Test improvement: +1.90%


In [11]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = ['Dev', 'Test']
base_accs = [results['Dev Baseline'], results['Test Baseline']]
innov_accs = [results['Dev Enhanced'], results['Test Enhanced']]

width = 0.35
x_pos = range(len(x))
bars1 = ax.bar([p - width/2 for p in x_pos], base_accs, width, label='Baseline', color='#3498db')
bars2 = ax.bar([p + width/2 for p in x_pos], innov_accs, width, label='Enhanced (+ Innovation)', color='#e74c3c')

ax.set_ylabel('Accuracy')
ax.set_title('Baseline vs Enhanced Tagger Accuracy')
ax.set_xticks(x_pos)
ax.set_xticklabels(x)
ax.legend()
ax.set_ylim(0.7, 1.0)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
            ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/baseline_vs_enhanced.png', dpi=150)
plt.show()

/tmp/ipykernel_1015481/1740455899.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Per-Tag Comparison

In [12]:
# Per-tag F1 comparison on dev set
base_metrics = Evaluator.per_tag_metrics(dev_tagged_base)
innov_metrics = Evaluator.per_tag_metrics(dev_tagged_innov)

comparison_rows = []
for tag in UPOS_TAGS:
    b = base_metrics.get(tag, {})
    e = innov_metrics.get(tag, {})
    comparison_rows.append({
        'Tag': tag,
        'Base F1': b.get('f1', 0),
        'Enhanced F1': e.get('f1', 0),
        'F1 Change': e.get('f1', 0) - b.get('f1', 0),
        'Support': b.get('support', 0),
    })

df_cmp = pd.DataFrame(comparison_rows).sort_values('F1 Change', ascending=False)
print(df_cmp.to_string(index=False))

  Tag  Base F1  Enhanced F1  F1 Change  Support
 PART   0.7907       0.9298     0.1391      647
 INTJ   0.5347       0.6452     0.1105      115
  AUX   0.8889       0.9426     0.0537     1567
  NUM   0.8841       0.9344     0.0503      383
PROPN   0.7681       0.8124     0.0443     1867
  ADP   0.8737       0.9085     0.0348     2038
 VERB   0.8342       0.8584     0.0242     2707
 NOUN   0.8378       0.8615     0.0237     4210
  DET   0.9386       0.9483     0.0097     1900
SCONJ   0.6340       0.6349     0.0009      397
PUNCT   0.9836       0.9844     0.0008     3075
  ADV   0.8277       0.8283     0.0006     1232
CCONJ   0.9741       0.9741     0.0000      779
  SYM   0.3563       0.3563     0.0000       81
 PRON   0.9483       0.9483     0.0000     2225
  ADJ   0.8720       0.8696    -0.0024     1865
    X   0.1039       0.0567    -0.0472       59


In [13]:
# F1 change chart
df_plot = df_cmp.sort_values('F1 Change')
colors = ['#27ae60' if x > 0 else '#e74c3c' for x in df_plot['F1 Change']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(df_plot['Tag'], df_plot['F1 Change'], color=colors)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('F1 Change (Enhanced - Baseline)')
ax.set_title('Per-Tag F1 Change with Innovation Modules')
plt.tight_layout()
plt.savefig('../outputs/innovation_f1_change.png', dpi=150)
plt.show()

/tmp/ipykernel_1015481/202873065.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Save Final Results

In [14]:
# Save final results
evaluator = Evaluator()
final_results = evaluator.run_full_evaluation(dev_tagged_innov, label='final', output_dir='../outputs')
print(f'Final accuracy (dev): {final_results["overall_accuracy"]}')

# Summary comparison
summary = {
    'baseline': {
        'dev_accuracy': round(Evaluator.accuracy(dev_tagged_base), 4),
        'test_accuracy': round(Evaluator.accuracy(test_tagged_base), 4),
    },
    'enhanced': {
        'dev_accuracy': round(Evaluator.accuracy(dev_tagged_innov), 4),
        'test_accuracy': round(Evaluator.accuracy(test_tagged_innov), 4),
    },
}
with open('../outputs/results_final.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved outputs/results_final.json')

Final accuracy (dev): 0.8934
Saved outputs/results_final.json
